<a href="https://colab.research.google.com/github/hwangho-kim/Transformer_Fewshot_PdM/blob/main/%EB%B0%98%EB%8F%84%EC%B2%B4_FDC_%EC%9D%B4%EC%83%81_%ED%83%90%EC%A7%80_%EC%A0%84%EC%B2%B4_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_R11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import time
import os
import math

# matplotlib 설정 (스타일 지정, 한글 폰트 배제)
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# [수정됨] 폰트 사이즈 글로벌 통일 (15)
plt.rcParams.update({
    'font.size': 15,
    'axes.titlesize': 15,
    'axes.labelsize': 15,
    'legend.fontsize': 13,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12
})

print("=====================================================")
print("=== Phase 1 & 2: Load Real CSV & Preprocessing    ===")
print("=====================================================")

# 1. 특정 컬럼만 읽어서 데이터프레임 생성
use_cols = ['ts', 'step_id', 'org_wf_id', 'param_alias', 'param_value']
try:
    df_raw = pd.read_csv('sample.csv', usecols=use_cols)
    print(f"-> Successfully loaded 'sample.csv'. Total records: {len(df_raw):,}")
except FileNotFoundError:
    print("-> [에러] 'sample.csv' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
    raise

# 2. 파이프라인 내부 변수명으로 매핑
df_raw = df_raw.rename(columns={
    'ts': 'TIMESTAMP',
    'step_id': 'STEP_ID',
    'org_wf_id': 'WAFER_ID',
    'param_alias': 'PARA_NAME',
    'param_value': 'VALUE'
})

df_raw = df_raw.sort_values(by=['WAFER_ID', 'TIMESTAMP'])

# 3. Wafer와 Parameter 자동 탐지
first_timestamps = df_raw.groupby('WAFER_ID')['TIMESTAMP'].min().sort_values()
detected_wafers = first_timestamps.index.tolist()
detected_params = list(df_raw['PARA_NAME'].unique())
num_params = len(detected_params)
num_wafers = len(detected_wafers)

print(f"-> Auto-detected Wafers: {num_wafers} ea")
print(f"-> Auto-detected Parameters: {num_params} ea")

# 4. Long to Wide Format (Pivot) & 스텝 전환 시점 기록
print("-> Converting Long format to Wide format & Handling missing values...")
df_raw['RELATIVE_TIME'] = df_raw.groupby(['WAFER_ID', 'PARA_NAME']).cumcount()

step_transitions = {}
for w_id in detected_wafers:
    w_df = df_raw[df_raw['WAFER_ID'] == w_id].drop_duplicates(subset=['STEP_ID'], keep='first')
    step_transitions[w_id] = w_df['RELATIVE_TIME'].tolist()

df_wide = df_raw.pivot(index=['WAFER_ID', 'RELATIVE_TIME'], columns='PARA_NAME', values='VALUE').reset_index()

# 5. 3D Tensor 구성
seq_len = df_wide['RELATIVE_TIME'].max() + 1
tensor_data = np.zeros((num_wafers, seq_len, num_params))

for w_idx, w_id in enumerate(detected_wafers):
    w_data = df_wide[df_wide['WAFER_ID'] == w_id].sort_values('RELATIVE_TIME')
    current_len = len(w_data)

    # ffill() 메서드를 직접 사용하여 Warning 제거
    w_vals = w_data[detected_params].ffill().fillna(0).values
    tensor_data[w_idx, :current_len, :] = w_vals

    if current_len < seq_len and current_len > 0:
        tensor_data[w_idx, current_len:, :] = w_vals[-1, :]

params_list = detected_params

# 6. 정규화 (시간순 앞부분 50%를 정상 상태로 정의 - 이로 인해 후반부 노후화 탐지 가능)
train_split_idx = max(1, int(num_wafers * 0.5))
train_data = tensor_data[:train_split_idx]

mean_val = train_data.mean(axis=(0, 1))
std_val = train_data.std(axis=(0, 1))

print("\n[Training Data Statistics - Selected Sensors (Top 4)]")
for i in range(min(4, num_params)):
    print(f" - {params_list[i]:>15}: Mean = {mean_val[i]:6.1f}, Std = {std_val[i]:5.1f}")

def normalize(data):
    return (data - mean_val) / (std_val + 1e-7)

def denormalize(data):
    return data * (std_val + 1e-7) + mean_val

tensor_data_norm = normalize(tensor_data)
train_data_norm = normalize(train_data)


print("\n=====================================================")
print("=== Phase 3: High-Performance Model Building      ===")
print("=====================================================")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# [혁신적 구조 개선] Channel-Independent Transformer 아키텍처 적용
class TS_MAE_CI(nn.Module):
    def __init__(self, seq_len, d_model=128, nhead=4, num_layers=3, mask_ratio=0.20):
        super().__init__()
        self.mask_ratio = mask_ratio

        # 센서들을 한꺼번에 섞지 않고, 1차원(스칼라) 단위로 개별 임베딩합니다.
        self.input_proj = nn.Linear(1, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=512, dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, 1)

    def forward(self, x):
        # x shape: (Batch, Seq_Len, Features)
        B, L, M = x.shape

        # [핵심] 채널(센서) 간 독립성 보장: Batch와 Feature를 합쳐서 개별 시계열로 취급
        # Shape 변환: (B * M, L, 1)
        x_ci = x.transpose(1, 2).reshape(B * M, L, 1)

        if self.mask_ratio > 0.0 and self.training:
            mask = torch.rand(B * M, L).to(x.device) < self.mask_ratio
            x_masked = x_ci.clone()
            x_masked[mask] = 0.0
        else:
            x_masked = x_ci

        encoded = self.input_proj(x_masked)
        encoded = self.pos_encoder(encoded)
        out = self.transformer(encoded)
        recon_ci = self.output_proj(out) # (B * M, L, 1)

        # 원래의 형태 (Batch, Seq_Len, Features)로 복원
        recon = recon_ci.reshape(B, M, L).transpose(1, 2)
        return recon

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 독립 채널 모델 인스턴스화
model = TS_MAE_CI(seq_len=seq_len, mask_ratio=0.20).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"-> Model Architecture: Channel-Independent Transformer MAE")
print(f"-> Trainable Parameters: {total_params:,}")

if torch.cuda.device_count() > 1:
    print(f"-> Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
else:
    print(f"-> Running on: {device}")

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.SmoothL1Loss() # 이상치 노이즈에 강건한 Smooth L1 Loss로 변경

train_tensor = torch.FloatTensor(train_data_norm).to(device)
epochs = 80
batch_size = 16

model.train()
print("\nStarting Pre-training on Golden Wafers (First 50%)...")
loss_history = []

start_train_time = time.time()
for epoch in range(epochs):
    epoch_loss = 0
    indices = torch.randperm(len(train_tensor))
    train_tensor_shuffled = train_tensor[indices]

    for i in range(0, len(train_tensor), batch_size):
        batch = train_tensor_shuffled[i:i+batch_size]
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / (len(train_tensor)/batch_size)
    loss_history.append(avg_loss)
    if (epoch+1) % 20 == 0:
        print(f" - Epoch [{epoch+1:>3}/{epochs}], Smooth L1 Loss: {avg_loss:.4f}")

print(f"-> Training Complete. ({time.time() - start_train_time:.2f} sec)")

# [수정됨] 차트 규격 통일 (12, 6)
plt.figure(figsize=(12, 6))
plt.plot(loss_history, color='navy', linewidth=2)
plt.title('Self-Supervised Pre-training Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Reconstruction Smooth L1 Loss')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 4: Unsupervised Inference & Scatter Plot ===")
print("=====================================================")
def calculate_anomaly_scores(model, data_tensor):
    model.eval()
    with torch.no_grad():
        x = torch.FloatTensor(data_tensor).to(device)
        recon = model(x)
        error_map = (x - recon) ** 2
        wafer_scores = torch.mean(error_map, dim=(1, 2)).cpu().numpy()
    return wafer_scores, error_map.cpu().numpy(), recon.cpu().numpy()

all_data_tensor = tensor_data_norm
anomaly_scores, error_maps, recon_data_norm = calculate_anomaly_scores(model, all_data_tensor)

train_scores = anomaly_scores[:100] if len(anomaly_scores) >= 100 else anomaly_scores[:train_split_idx]

# [수정됨] 2-Sigma (경고), 3-Sigma (위험) 임계치 분리 설정
mean_score = np.mean(train_scores)
std_score = np.std(train_scores)
threshold_2sig = mean_score + 2 * std_score
threshold_3sig = mean_score + 3 * std_score

print(f"-> Normal Data Mean Score: {mean_score:.4f}")
print(f"-> Normal Data Std Score:  {std_score:.4f}")
print(f"-> Set Warning Threshold (Mean + 2*Std): {threshold_2sig:.4f}")
print(f"-> Set Critical Threshold (Mean + 3*Std): {threshold_3sig:.4f}")

wafer_indices = np.arange(1, num_wafers + 1)

# [수정됨] 조건에 따른 웨이퍼 분류
is_normal = anomaly_scores <= threshold_2sig
is_warning = (anomaly_scores > threshold_2sig) & (anomaly_scores <= threshold_3sig)
is_critical = anomaly_scores > threshold_3sig

# [수정됨] 차트 규격 통일 (12, 6) 및 다중 임계치 표현
plt.figure(figsize=(12, 6))
plt.scatter(wafer_indices[is_normal], anomaly_scores[is_normal],
            color='dodgerblue', alpha=0.6, label='Normal (<= 2σ)')
plt.scatter(wafer_indices[is_warning], anomaly_scores[is_warning],
            color='orange', marker='^', s=100, linewidth=2, label='Warning (> 2σ)')
plt.scatter(wafer_indices[is_critical], anomaly_scores[is_critical],
            color='red', marker='x', s=100, linewidth=2, label='Critical (> 3σ)')

plt.axhline(y=threshold_2sig, color='orange', linestyle='--', label=f'2σ Threshold ({threshold_2sig:.4f})')
plt.axhline(y=threshold_3sig, color='red', linestyle='--', label=f'3σ Threshold ({threshold_3sig:.4f})')

plt.title('Unsupervised Equipment Health Monitoring: Wafer Anomaly Scatter')
plt.xlabel('Wafer Processing Sequence')
plt.ylabel('Anomaly Score (Reconstruction Error)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


print("\n=====================================================")
print("=== Phase 5: Auto-Triggered Root Cause Analysis   ===")
print("=====================================================")
def generate_business_report(target_idx, rank):
    w_id = detected_wafers[target_idx]
    score = anomaly_scores[target_idx]
    e_map = error_maps[target_idx]

    print(f"\n[{rank} Rank Anomaly] RCA Report Generated for {w_id} (Score: {score:.4f})")

    # [수정됨] 히트맵 비율 및 크기 최적화 (가독성 향상, 높이 1.2배 증가)
    heatmap_height = max(4.8, num_params * 0.48)
    plt.figure(figsize=(12, heatmap_height))
    sns.heatmap(e_map.T, cmap='Reds', cbar_kws={'label': 'Error Magnitude'})
    plt.yticks(ticks=np.arange(num_params)+0.5, labels=params_list, rotation=0, fontsize=8)
    plt.title(f'Root Cause Analysis X-Ray (Target: {w_id} - Auto Detected)', fontsize=12)
    plt.xlabel('Process Time (sec) / Step Progression', fontsize=10)

    transitions = step_transitions.get(w_id, [])
    for t_idx in transitions:
        if t_idx > 0:
            plt.axvline(t_idx, color='blue', linestyle=':', alpha=0.5)

    plt.tight_layout()
    plt.show()

    actual_trace = denormalize(all_data_tensor[target_idx])
    recon_trace = denormalize(recon_data_norm[target_idx])

    num_rows = (num_params + 1) // 2
    # [수정됨] Trace 서브플롯의 세로 길이 축소로 전체 스크롤 부담 완화
    fig, axes = plt.subplots(num_rows, 2, figsize=(14, max(4, 1.8 * num_rows)), sharex=True)
    if num_rows > 1:
        axes = axes.flatten()
    else:
        axes = np.array(axes).flatten()

    for p_idx, param_name in enumerate(params_list):
        axes[p_idx].plot(actual_trace[:, p_idx], label='Actual Sensor Value', color='black', linewidth=1.2)
        axes[p_idx].plot(recon_trace[:, p_idx], label='AI Expected Value', color='red', linestyle='--', linewidth=1.2)

        # [수정됨] 수평선 임계치 0 방지 로직 (Base Noise Level 추가)
        trace_std = np.std(actual_trace[:, p_idx])
        trace_range = np.max(actual_trace[:, p_idx]) - np.min(actual_trace[:, p_idx])
        base_tolerance = max(np.mean(np.abs(actual_trace[:, p_idx])) * 0.02, 0.5)
        dynamic_threshold = max(trace_std * 0.5, trace_range * 0.1, base_tolerance)

        diff = np.abs(actual_trace[:, p_idx] - recon_trace[:, p_idx])
        axes[p_idx].fill_between(range(seq_len), 0, 1, where=(diff > dynamic_threshold),
                             color='red', alpha=0.2, transform=axes[p_idx].get_xaxis_transform())

        axes[p_idx].set_title(param_name, fontsize=10, pad=3)
        if p_idx == 0:
            axes[p_idx].legend(loc='upper right', fontsize=8)

        for t_idx in transitions:
            if t_idx > 0:
                axes[p_idx].axvline(t_idx, color='blue', linestyle='--', alpha=0.5)

    for i in range(num_params, len(axes)):
        fig.delaxes(axes[i])

    for i in range(max(0, num_params - 2), num_params):
        axes[i].set_xlabel('Process Time (sec)', fontsize=12)

    plt.suptitle(f'Detailed Sensor Analysis: Actual vs AI Expectation ({w_id})', fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

# [수정됨] 2-Sigma 이상인 모든 웨이퍼를 대상으로 RCA 생성
detected_anomalies = np.where(anomaly_scores > threshold_2sig)[0]
print(f"-> Total Anomalies Detected > 2-Sigma Threshold: {len(detected_anomalies)}")

if len(detected_anomalies) > 0:
    sorted_anomalies = detected_anomalies[np.argsort(anomaly_scores[detected_anomalies])[::-1]]

    print(f"-> Generating RCA reports for ALL {len(sorted_anomalies)} anomalies in descending order...")

    for rank, target_idx in enumerate(sorted_anomalies):
        generate_business_report(target_idx, rank + 1)
else:
    print("-> No anomalies detected above the 2-Sigma threshold.")


print("\n=====================================================")
print("=== Phase 6: Future Trend Prediction (Prognostics) ===")
print("=====================================================")
recent_window = max(20, int(num_wafers * 0.5)) # ARIMA 학습을 위해 충분한 윈도우 확보
recent_wafers = wafer_indices[-recent_window:]
recent_scores = anomaly_scores[-recent_window:]

future_horizon = max(10, int(num_wafers * 0.25))
future_wafers = np.arange(num_wafers + 1, num_wafers + future_horizon + 1)

try:
    from statsmodels.tsa.arima.model import ARIMA
    import warnings
    warnings.filterwarnings("ignore") # 상태 공간 모델 최적화 경고 숨김

    print(f"-> Forecasting with ARIMA (AutoRegressive Integrated Moving Average)")

    # ARIMA(p,d,q) 모델 피팅
    # - p=1: 이전 1스텝의 자기상관성 반영
    # - d=1: 데이터의 우상향 추세(Trend)를 잡기 위한 1차 차분
    # - q=1: 이전 1스텝의 예측 오차 반영
    arima_model = ARIMA(recent_scores, order=(1, 1, 1))
    arima_result = arima_model.fit()

    # 과거 구간 추세선 (In-sample prediction)
    smoothed_scores = arima_result.predict(start=0, end=len(recent_scores)-1)
    smoothed_scores[0] = recent_scores[0] # 첫 번째 예측값 보정

    # 미래 구간 예측 (Out-of-sample forecast)
    future_predictions = arima_result.forecast(steps=future_horizon)

    eq_str = "Trend Method: ARIMA(1, 1, 1)\n(Time-Series Forecasting)"

except ImportError:
    print("-> [알림] 'statsmodels' 라이브러리가 없어 EWMA 모델로 대체합니다. (pip install statsmodels 필요)")
    alpha = 0.3
    beta = 0.1
    level = np.zeros(len(recent_scores))
    trend = np.zeros(len(recent_scores))
    level[0] = recent_scores[0]
    trend[0] = recent_scores[1] - recent_scores[0] if len(recent_scores) > 1 else 0

    for i in range(1, len(recent_scores)):
        level[i] = alpha * recent_scores[i] + (1 - alpha) * (level[i-1] + trend[i-1])
        trend[i] = beta * (level[i] - level[i-1]) + (1 - beta) * trend[i-1]

    smoothed_scores = level
    future_predictions = level[-1] + np.arange(1, future_horizon + 1) * trend[-1]
    eq_str = f"Trend Method: Double EWMA (Holt's)\n$\\alpha$={alpha}, $\\beta$={beta}"

# [수정됨] 차트 규격 통일 (12, 6)
plt.figure(figsize=(12, 6))

plt.scatter(wafer_indices, anomaly_scores, color='lightgray', s=30, label='Actual Anomaly Scores')
plt.plot(recent_wafers, smoothed_scores, color='blue', linewidth=2, label='Smoothed Trend (In-sample)')
plt.plot(future_wafers, future_predictions, color='red', linestyle='--', linewidth=3.0, label='Predicted Future Trend')

# 임계치 2시그마, 3시그마로 매핑
plt.axhline(y=threshold_2sig, color='orange', linestyle='--', linewidth=1.5, label=f'2σ Warning ({threshold_2sig:.2f})')
plt.axhline(y=threshold_3sig, color='red', linestyle='-', linewidth=1.5, label=f'3σ Critical ({threshold_3sig:.2f})')

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

plt.text(0.02, 0.85, eq_str, transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

score_desc = ("* Anomaly Score: 모델이 복원한 정상 패턴과 실제 센서 데이터 간의 평균 복원 오차(MSE).\n"
              "  값이 클수록 장비가 기준 베이스라인(건강한 상태)에서 심각하게 벗어나고 있음을 의미합니다.")
plt.text(0.02, 0.05, score_desc, transform=plt.gca().transAxes, fontsize=12,
         bbox=dict(facecolor='lightyellow', alpha=0.9, edgecolor='orange'))

plt.title('Equipment Prognostics: Future Anomaly Trend Prediction')
plt.xlabel('Wafer Processing Sequence (Including Future)')
plt.ylabel('Anomaly Score')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print("\n=== Pipeline Execution Completed Successfully ===")